## 📑 PageIndex — Vectorless RAG with Claude Haiku
### Reasoning-based RAG, no vector DB, no chunking
**EN version — essential comments only**

---

### 🔑 Key Concept

> **Traditional RAG** → chunk → embed → cosine similarity → retrieve
> **PageIndex RAG** → build tree → LLM reasons over tree → retrieve exact sections

`Similarity ≠ Relevance` — a chunk about "market conditions" may score higher
than the real answer section just because it shares more words with the query.

---
### Section 1: Install & Setup

- Load API keys from `.env`
- Initialize both clients:
  - 🔑 PageIndex key: https://dash.pageindex.ai/api-keys
  - 🔑 Anthropic key: https://console.anthropic.com/settings/keys

In [14]:
# ── Create a .env file (run this once) ──────────────────────────────────────
# env_content = """
# PAGEINDEX_API_KEY=your_pageindex_key_here
# ANTHROPIC_API_KEY=your_anthropic_key_here
# """
# with open(".env", "w") as f:
#     f.write(env_content.strip())
# print("✅ .env file created")

In [15]:
import os, json, time
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY = os.getenv("PAGEINDEX_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("Anthropic key loaded:", "✅" if ANTHROPIC_API_KEY else "❌ Missing!")

PageIndex key loaded: ✅
Anthropic key loaded: ✅


In [16]:
from pageindex import PageIndexClient
from anthropic import Anthropic

pi_client     = PageIndexClient(api_key=PAGEINDEX_API_KEY)
claude_client = Anthropic(api_key=ANTHROPIC_API_KEY)

# Fast, low-cost Claude model — a good fit for tree-search + answer generation
CLAUDE_MODEL = "claude-haiku-4-5-20251001"

print("✅ PageIndex client ready")
print("✅ Anthropic client ready")
print(f"   Model: {CLAUDE_MODEL}")

✅ PageIndex client ready
✅ Anthropic client ready
   Model: claude-haiku-4-5-20251001


---
### Section 2: Upload & Index a PDF

1. Upload your PDF to the PageIndex cloud
2. PageIndex uses an LLM to read the document structure
3. Builds a hierarchical **tree index** (like a smart Table of Contents)
4. Returns a `doc_id` for all future operations

In [28]:
# Path to the sample PDF file
PDF_PATH = "./data/sample_document.pdf"

print(f"Uploading: {PDF_PATH}")
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]

print("✅ Uploaded!")
print(f"📋 Document ID: {doc_id}")

Uploading: ./data/sample_document.pdf
✅ Uploaded!
📋 Document ID: pi-cmrxhjwd9003z01p5p7rq3ycc


In [27]:
# Poll until processing is complete (async build — 30-90s for ~50 pages)
print("Building tree index...")

while True:
    status_result = pi_client.get_document(doc_id)
    status = status_result.get("status")
    print(f"   Status: {status}")

    if status == "completed":
        print("\n✅ Tree index ready!")
        break
    elif status == "failed":
        print("\n❌ Processing failed. Check your PDF format.")
        break

    time.sleep(5)

Building tree index...
   Status: completed

✅ Tree index ready!


---
### Section 3: Inspect the Tree Structure

Each node has: `node_id`, `title`, `page_index`, `text`/`summary`, `nodes` (children).
This structure is what the LLM reasons over during retrieval.

In [19]:
tree_result    = pi_client.get_tree(doc_id, node_summary=True)
pageindex_tree = tree_result.get("result", [])

print(f"Top-level sections: {len(pageindex_tree)}")
print("\nRaw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top-level sections: 1

Raw tree (first node):
{
  "title": "NovaTerra Cloud Systems, Inc.",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# NovaTerra Cloud Systems, Inc.\n\nAnnual Report & Operations Review \u2014 Fiscal Year 2025\n\nA fictional company, created solely as a sample document for retrieval demos\n",
  "text": "# NovaTerra Cloud Systems, Inc.\n\nAnnual Report & Operations Review \u2014 Fiscal Year 2025\n\nA fictional company, created solely as a sample document for retrieval demos\n",
  "nodes": [
    {
      "title": "1. Executive Summary",
      "node_id": "0001",
      "page_index": 2,
      "summary": "NovaTerra Cloud Systems reported strong fiscal 2025 results with $482 million in revenue and improved operating margins driven by its Managed Storage and AI Compute segments. The company is aggressively expanding its data center infrastructure to meet long-term AI demand, while balancing this growth strategy against risks related to capital intensity and h

In [20]:
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("Full Document Structure:\n")
print_tree(pageindex_tree)

Full Document Structure:

[0000] NovaTerra Cloud Systems, Inc.  (p.1)
  └─ [0001] 1. Executive Summary  (p.2)
  └─ [0002] 2. Company Overview  (p.2)
  └─ [0003] 3. Financial Performance  (p.2)
  └─ [0004] 4. Risk Factors  (p.3)
  └─ [0005] 5. Technology and Infrastructure  (p.4)
  └─ [0006] 6. Human Resources  (p.4)
  └─ [0007] 7. Outlook and Strategy  (p.4)
  └─ [0008] 8. Conclusion  (p.5)


---
### Section 4: LLM Tree Search — The Core of PageIndex

```
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
```

Note: unlike the OpenAI client, the Anthropic API has no built-in
`response_format=json_object` mode — we explicitly ask the model to reply
with raw JSON and parse the text ourselves.

In [21]:
def llm_tree_search(query: str, tree: list, model: str = CLAUDE_MODEL) -> dict:
    """
    Core PageIndex retrieval: sends query + tree to Claude, gets back
    relevant node_ids with reasoning.
    """

    # Compress tree to save tokens — titles + short summaries only
    def compress(nodes):
        out = []
        for n in nodes:
            entry = {
                "node_id": n["node_id"],
                "title":   n["title"],
                "page":    n.get("page_index", "?"),
                "summary": n.get("text", "")[:150]
            }
            if n.get("nodes"):
                entry["children"] = compress(n["nodes"])
            out.append(entry)
        return out

    compressed_tree = compress(tree)

    prompt = f"""You are given a query and a document's tree structure (like a Table of Contents).
Identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{json.dumps(compressed_tree, indent=2)}

Reply ONLY with raw JSON, no extra text, in this exact format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}"""

    response = claude_client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )

    raw_text = response.content[0].text.strip()
    raw_text = raw_text.replace("```json", "").replace("```", "").strip()

    return json.loads(raw_text)

In [22]:
query = "What topics does this document cover?"

print(f"Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("Selected Node IDs:", result.get("node_list", []))

Query: What topics does this document cover?

LLM Reasoning:
The query asks 'What topics does this document cover?' This is a request for an overview of the document's content. The most comprehensive way to answer this would be to look at the structure of the entire document. The root node (0000) provides the title and indicates this is an Annual Report & Operations Review. To give a complete answer about all topics covered, I should reference the main section headers which directly list the topics. The Executive Summary (0001) would likely provide a high-level overview of topics covered. However, the most direct answer comes from looking at all the main sections listed under the root node: sections 1-8 represent the topics covered. Since the query is asking what topics are covered overall, I should identify the node that best lists or describes all topics. The root node structure itself shows all the topics, but if looking for a narrative description, the Executive Summary (0001) woul

---
### Section 5: Full End-to-End RAG Pipeline

1. **Tree Search** → Claude picks relevant `node_ids`
2. **Retrieve** → fetch actual section content
3. **Generate** → Claude writes a grounded, cited answer

In [23]:
def find_nodes_by_ids(tree: list, target_ids: list) -> list:
    """Recursively walk the tree and collect nodes matching target_ids."""
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        if node.get("nodes"):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))
    return found

In [24]:
def generate_answer(query: str, nodes: list, model: str = CLAUDE_MODEL) -> str:
    """Takes retrieved nodes as context and generates a cited, grounded answer."""
    if not nodes:
        return "⚠️ No relevant sections found in the document."

    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:"""

    response = claude_client.messages.create(
        model=model,
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )

    return response.content[0].text

In [25]:
def vectorless_rag(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print("=" * 55)
        print(f"Query: {query}")
        print("=" * 55)

    search_result = llm_tree_search(query, tree)
    node_ids      = search_result.get("node_list", [])

    if verbose:
        print(f"\nReasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"Retrieved node IDs: {node_ids}")

    nodes = find_nodes_by_ids(tree, node_ids)

    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")

    answer = generate_answer(query, nodes)

    if verbose:
        print(f"\nAnswer:\n{answer}")

    return answer

In [26]:
test_queries = [
    "What are the main topics covered in this document?",
    "Summarize the key takeaways from the document.",
]

for q in test_queries:
    print()
    ans = vectorless_rag(q, pageindex_tree, verbose=False)
    print(f"Q: {q}")
    print(f"A: {ans[:400]}...")
    print("-" * 55)


Q: What are the main topics covered in this document?
A: # Main Topics Covered in This Document

The document covers the following main topics:

1. **Executive Summary** – NovaTerra's fiscal year 2025 financial performance, including $482 million in revenue (18% increase) and 21.4% operating margin, along with infrastructure expansion plans (1. Executive Summary, Page 2)

2. **Company Overview** – General information about the company (2. Company Overvi...
-------------------------------------------------------

Q: Summarize the key takeaways from the document.
A: # Key Takeaways

**Financial Performance:**
NovaTerra achieved $482 million in revenue with 18% year-over-year growth, driven by Managed Storage and AI Compute segments. Operating margin improved to 21.4% from 17.9%, reflecting improved data center efficiency and cost discipline. (Executive Summary, Page 2)

**Strategic Investments:**
The Company invested heavily in infrastructure, opening two new...
------------------------

---
### Summary

Built a complete **Vectorless RAG** system using PageIndex and **Claude Haiku 4.5**.

- **`llm_tree_search()`** — Claude reasons over the document tree to find relevant nodes
- **`find_nodes_by_ids()`** — retrieves actual section content
- **`generate_answer()`** — Claude produces a cited, grounded answer
- **`vectorless_rag()`** — full pipeline combining all 3 steps

`Similarity ≠ Relevance` remains the fundamental flaw of vector search that tree-based
reasoning avoids. Claude Haiku is a good fit here: fast and low-cost, well suited to
tree-navigation and short grounded-answer generation.

---

### 🔗 Resources
- PageIndex GitHub: https://github.com/VectifyAI/PageIndex
- PageIndex Docs: https://docs.pageindex.ai
- Anthropic Docs: https://docs.claude.com